# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [6]:

from pathlib import Path
import duckdb
import polars as pl

# Tentukan root project
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Lokasi dataset sesuai struktur folder
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"

# Lokasi output hasil cleaning
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

print("Lokasi dataset:", DATA_PATH)
print("File ditemukan:", DATA_PATH.exists())

# Baca kedua sheet Excel
df_2009 = pl.read_excel(
    DATA_PATH,
    sheet_name="Year 2009-2010",
    engine="calamine",
    infer_schema_length=None
)

df_2010 = pl.read_excel(
    DATA_PATH,
    sheet_name="Year 2010-2011",
    engine="calamine",
    infer_schema_length=None
)

# Samakan tipe data
kolom_string = ["Invoice", "StockCode", "Customer ID"]

df_2009 = df_2009.with_columns(
    [pl.col(k).cast(pl.String) for k in kolom_string]
)

df_2010 = df_2010.with_columns(
    [pl.col(k).cast(pl.String) for k in kolom_string]
)

# Gabungkan dataset
df = pl.concat(
    [df_2009, df_2010],
    how="vertical_relaxed"
)

print(f"Jumlah baris: {df.height:,}")
print(f"Jumlah kolom: {df.width}")

df.head()


Lokasi dataset: C:\Tugas\All About Praktikum\SEMESTER 7 INFORMATIKA\ANALISIS BIG DATA\tugas1-abd-2026\data\raw\online_retail_II.xlsx
File ditemukan: True
Jumlah baris: 1,067,371
Jumlah kolom: 8


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
str,str,str,i64,datetime[ms],f64,str,str
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom"""
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom"""
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [9]:
# Profiling dataset menggunakan DuckDB
connection = duckdb.connect()

connection.register("dataset", df)

# Jalankan profiling
profil = pl.from_pandas(
    connection.execute("""
        SUMMARIZE SELECT * FROM dataset
    """).df()
)

print("HASIL DATA PROFILING")
print(profil)


HASIL DATA PROFILING
shape: (8, 12)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬─────────┬───────────┐
│ column_nam ┆ column_ty ┆ min       ┆ max       ┆ … ┆ q50       ┆ q75       ┆ count   ┆ null_perc │
│ e          ┆ pe        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---     ┆ entage    │
│ ---        ┆ ---       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ i64     ┆ ---       │
│ str        ┆ str       ┆           ┆           ┆   ┆           ┆           ┆         ┆ f64       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═════════╪═══════════╡
│ Invoice    ┆ VARCHAR   ┆ 489434    ┆ C581569   ┆ … ┆ null      ┆ null      ┆ 1067371 ┆ 0.0       │
│ StockCode  ┆ VARCHAR   ┆ 10002     ┆ m         ┆ … ┆ null      ┆ null      ┆ 1067371 ┆ 0.0       │
│ Descriptio ┆ VARCHAR   ┆ DOORMAT   ┆ wrongly   ┆ … ┆ null      ┆ null      ┆ 1067371 ┆ 0.41      │
│ n          ┆           ┆ UNION     ┆ sold sets ┆   ┆ 

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [11]:

# Hitung jumlah baris
jumlah_baris = len(df)

# Deteksi missing values pada semua kolom
null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (
            pl.col("jumlah_null") / jumlah_baris * 100
        ).round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)

print("RINGKASAN MISSING VALUES SEMUA KOLOM")
print(null_summary)


RINGKASAN MISSING VALUES SEMUA KOLOM
shape: (8, 3)
┌─────────────┬─────────────┬─────────────┐
│ kolom       ┆ jumlah_null ┆ persen_null │
│ ---         ┆ ---         ┆ ---         │
│ str         ┆ u32         ┆ f64         │
╞═════════════╪═════════════╪═════════════╡
│ Customer ID ┆ 243007      ┆ 22.77       │
│ Description ┆ 4382        ┆ 0.41        │
│ Invoice     ┆ 0           ┆ 0.0         │
│ StockCode   ┆ 0           ┆ 0.0         │
│ Quantity    ┆ 0           ┆ 0.0         │
│ InvoiceDate ┆ 0           ┆ 0.0         │
│ Price       ┆ 0           ┆ 0.0         │
│ Country     ┆ 0           ┆ 0.0         │
└─────────────┴─────────────┴─────────────┘


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [12]:
# Penanganan missing values
df_missing_handled = df.with_columns([
    pl.col("Description")
      .fill_null("Unknown Product"),

    pl.col("Country")
      .fill_null("Unknown"),

    pl.col("Invoice")
      .fill_null("Unknown"),

    pl.col("StockCode")
      .fill_null("Unknown")
])

# Customer ID tidak diimputasi
# karena merupakan identitas pelanggan

print("HASIL PENANGANAN MISSING VALUES")

print(
    "Null sebelum:",
    df.null_count().sum_horizontal().sum()
)

print(
    "Null sesudah:",
    df_missing_handled.null_count().sum_horizontal().sum()
)

print(df_missing_handled.null_count())

df_missing_handled.head()


HASIL PENANGANAN MISSING VALUES
Null sebelum: 247389
Null sesudah: 243007
shape: (1, 8)
┌─────────┬───────────┬─────────────┬──────────┬─────────────┬───────┬─────────────┬─────────┐
│ Invoice ┆ StockCode ┆ Description ┆ Quantity ┆ InvoiceDate ┆ Price ┆ Customer ID ┆ Country │
│ ---     ┆ ---       ┆ ---         ┆ ---      ┆ ---         ┆ ---   ┆ ---         ┆ ---     │
│ u32     ┆ u32       ┆ u32         ┆ u32      ┆ u32         ┆ u32   ┆ u32         ┆ u32     │
╞═════════╪═══════════╪═════════════╪══════════╪═════════════╪═══════╪═════════════╪═════════╡
│ 0       ┆ 0         ┆ 0           ┆ 0        ┆ 0           ┆ 0     ┆ 243007      ┆ 0       │
└─────────┴───────────┴─────────────┴──────────┴─────────────┴───────┴─────────────┴─────────┘


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
str,str,str,i64,datetime[ms],f64,str,str
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom"""
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom"""
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
KOLOM_NUMERIK = "Price"

# Gunakan harga positif untuk menghitung IQR
harga_valid = df_missing_handled.filter(
    pl.col("Price") > 0
)

q1 = harga_valid["Price"].quantile(0.25)
q3 = harga_valid["Price"].quantile(0.75)

iqr = q3 - q1

batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Batas bawah: {batas_bawah}")
print(f"Batas atas: {batas_atas}")

# Deteksi outlier
outlier = df_missing_handled.filter(
    (pl.col("Price") > batas_atas) |
    (pl.col("Price") < 0)
)

print(f"Jumlah outlier: {outlier.height:,}")

# Winsorization pada harga positif
# Harga asli tetap disimpan untuk audit
df_outlier_handled = df_missing_handled.with_columns(
    pl.col("Price").alias("Price_Original")
).with_columns(
    pl.when(pl.col("Price") > 0)
      .then(
          pl.col("Price").clip(
              lower_bound=batas_bawah,
              upper_bound=batas_atas
          )
      )
      .otherwise(pl.col("Price"))
      .alias("Price")
)

df_outlier_handled.head()


Q1: 1.25
Q3: 4.15
IQR: 2.9000000000000004
Batas bawah: 0
Batas atas: 8.5
Jumlah outlier: 68,105


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Price_Original
str,str,str,i64,datetime[ms],f64,str,str,f64
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom""",6.95
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom""",6.75
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom""",6.75
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom""",2.1
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom""",1.25


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [14]:
KOLOM_KUNCI = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price_Original",
    "Customer ID",
    "Country"
]

total_baris = len(df_outlier_handled)

baris_unik = df_outlier_handled.select(
    KOLOM_KUNCI
).unique().height

jumlah_duplikat = total_baris - baris_unik

print(f"Total baris: {total_baris:,}")
print(f"Baris unik: {baris_unik:,}")
print(f"Duplikat: {jumlah_duplikat:,}")

# Hapus baris identik
df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep="first",
    maintain_order=True
)

print(f"Baris setelah deduplikasi: {len(df_dedup):,}")

df_dedup.head()


Total baris: 1,067,371
Baris unik: 1,033,036
Duplikat: 34,335
Baris setelah deduplikasi: 1,033,036


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Price_Original
str,str,str,i64,datetime[ms],f64,str,str,f64
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom""",6.95
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom""",6.75
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom""",6.75
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom""",2.1
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom""",1.25


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [17]:

# Buat koneksi baru agar tidak bentrok
conn_validasi = duckdb.connect(database=":memory:")

# Registrasi dataset hasil cleaning
conn_validasi.register("dataset_bersih", df_dedup)

# Validasi menggunakan DuckDB
hasil_query = conn_validasi.execute("""
    SELECT
        COUNT(*) AS total_baris,

        COUNT(*) FILTER (
            WHERE Price IS NULL
        ) AS pelanggaran_null,

        COUNT(*) FILTER (
            WHERE Price < 0
        ) AS pelanggaran_harga_negatif,

        COUNT(*) FILTER (
            WHERE Price > ?
        ) AS pelanggaran_batas_atas,

        COUNT(*) FILTER (
            WHERE Quantity = 0
        ) AS pelanggaran_quantity_nol,

        COUNT(*) FILTER (
            WHERE Quantity < 0
        ) AS transaksi_quantity_negatif,

        COUNT(*) FILTER (
            WHERE "Customer ID" IS NULL
        ) AS customer_id_kosong,

        COUNT(*) FILTER (
            WHERE InvoiceDate IS NULL
        ) AS tanggal_kosong

    FROM dataset_bersih
""", [batas_atas]).fetchdf()

# Konversi ke Polars
validasi = pl.from_pandas(hasil_query)

# Pemeriksaan duplikat
duplikat_tersisa = (
    len(df_dedup)
    - df_dedup.select(KOLOM_KUNCI).unique().height
)

# Tambahkan hasil duplikat
validasi = validasi.with_columns(
    pl.lit(duplikat_tersisa).alias("duplikat_tersisa")
)

print("HASIL VALIDASI DATA")
print(validasi)

# Hitung total pelanggaran utama
kolom_pelanggaran = [
    "pelanggaran_null",
    "pelanggaran_harga_negatif",
    "pelanggaran_batas_atas",
    "pelanggaran_quantity_nol",
    "tanggal_kosong",
    "duplikat_tersisa"
]

total_pelanggaran = sum(
    validasi[kolom][0]
    for kolom in kolom_pelanggaran
)

print("\nKESIMPULAN VALIDASI")

if total_pelanggaran == 0:
    print("VALIDASI BERHASIL")
    print("Tidak ditemukan pelanggaran utama.")
else:
    print(f"Ditemukan {total_pelanggaran:,} pelanggaran.")
    print("Beberapa data masih perlu ditinjau.")

print("\nCATATAN:")
print("- Quantity negatif dapat menunjukkan transaksi retur.")
print("- Customer ID kosong dipertahankan karena merupakan identitas pelanggan.")
print("- Harga negatif perlu diperiksa sebelum analisis penjualan.")

# Tutup koneksi setelah selesai
conn_validasi.close()


HASIL VALIDASI DATA
shape: (1, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ total_bar ┆ pelanggar ┆ pelanggar ┆ pelanggar ┆ … ┆ transaksi ┆ customer_ ┆ tanggal_k ┆ duplikat │
│ is        ┆ an_null   ┆ an_harga_ ┆ an_batas_ ┆   ┆ _quantity ┆ id_kosong ┆ osong     ┆ _tersisa │
│ ---       ┆ ---       ┆ negatif   ┆ atas      ┆   ┆ _negatif  ┆ ---       ┆ ---       ┆ ---      │
│ i64       ┆ i64       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ i64       ┆ i64       ┆ i32      │
│           ┆           ┆ i64       ┆ i64       ┆   ┆ i64       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1033036   ┆ 0         ┆ 5         ┆ 0         ┆ … ┆ 22496     ┆ 235151    ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘

KESIMPULAN VALIDASI
Ditemukan 5 pelanggaran.
Beberapa da

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [18]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_dedup.write_parquet(OUTPUT_PATH)

print("HASIL DATA CLEANING")
print(f"Baris sebelum: {len(df):,}")
print(f"Baris sesudah: {len(df_dedup):,}")
print(f"Baris berkurang: {len(df) - len(df_dedup):,}")
print(f"File tersimpan: {OUTPUT_PATH}")


HASIL DATA CLEANING
Baris sebelum: 1,067,371
Baris sesudah: 1,033,036
Baris berkurang: 34,335
File tersimpan: C:\Tugas\All About Praktikum\SEMESTER 7 INFORMATIKA\ANALISIS BIG DATA\tugas1-abd-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

## Catatan Milestone 2

### 1. Dimensi Kualitas Data

Dataset yang digunakan adalah Online Retail II, yang berisi data transaksi penjualan retail dari periode 2009–2011.

Dimensi kualitas data yang dianalisis meliputi:

- **Completeness:** Pemeriksaan missing values, terutama pada kolom `Customer ID` dan `Description`.
- **Validity:** Pemeriksaan nilai `Price`, `Quantity`, dan `InvoiceDate`.
- **Uniqueness:** Pemeriksaan duplikasi berdasarkan kombinasi atribut transaksi.
- **Consistency:** Pemeriksaan kesesuaian tipe data dan karakteristik transaksi.

### 2. Strategi Penanganan Missing Values

Missing values pada kolom `Description` diisi dengan nilai `"Unknown Product"`, sedangkan kolom `Country`, `Invoice`, dan `StockCode` diisi dengan `"Unknown"` apabila terdapat nilai kosong.

Missing values pada `Customer ID` tidak diimputasi menggunakan mean atau median karena kolom tersebut merupakan identitas pelanggan. Mengisi nilai tersebut secara sembarang dapat menyebabkan kesalahan identifikasi pelanggan.

### 3. Strategi Penanganan Outlier

Deteksi outlier dilakukan pada kolom `Price` menggunakan metode Interquartile Range (IQR).

Batas outlier ditentukan dengan rumus:

- IQR = Q3 − Q1
- Batas bawah = maksimum(0, Q1 − 1,5 × IQR)
- Batas atas = Q3 + 1,5 × IQR

Penanganan outlier dilakukan menggunakan teknik winsorization pada harga positif. Nilai harga asli dipertahankan pada kolom `Price_Original` untuk menjaga keterlacakan data.

### 4. Penanganan Duplikasi

Deduplikasi dilakukan berdasarkan kombinasi atribut transaksi, yaitu `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price_Original`, `Customer ID`, dan `Country`.

Kolom `Invoice` tidak digunakan sebagai satu-satunya kunci karena satu invoice dapat terdiri atas beberapa produk.

### 5. Aturan Validasi

Validasi data dilakukan dengan memeriksa:

- Nilai null pada kolom `Price` dan `InvoiceDate`.
- Harga negatif dan harga di atas batas IQR.
- Nilai `Quantity` sama dengan nol.
- Jumlah transaksi dengan `Quantity` negatif.
- Missing values pada `Customer ID`.
- Duplikasi yang masih tersisa setelah proses cleaning.

Nilai `Quantity` negatif tidak langsung dihapus karena dapat menunjukkan transaksi pembatalan atau retur.

### 6. Perubahan Jumlah Baris

Dataset awal terdiri atas 1.067.371 baris dari dua periode transaksi.

Jumlah baris setelah cleaning dan jumlah duplikat yang dihapus diperoleh dari hasil eksekusi Cell 11 dan Cell 15.

### 7. Penyimpanan Data

Dataset hasil cleaning disimpan dalam format Parquet dengan lokasi:

`data/processed/dataset_bersih.parquet`

Dataset tersebut selanjutnya dapat digunakan pada Milestone 3 tanpa harus mengulang seluruh proses data cleaning.

### Kesimpulan

Proses data cleaning dilakukan untuk meningkatkan kelengkapan, validitas, dan keunikan data transaksi. Penanganan missing values, outlier, dan duplikasi dilakukan dengan mempertimbangkan karakteristik dataset retail, termasuk transaksi retur dan identitas pelanggan yang tidak tersedia.

Hasil akhir berupa dataset yang lebih terstruktur dan siap digunakan untuk analisis lanjutan, dengan tetap mempertahankan informasi yang diperlukan untuk audit kualitas data.
